In [90]:
tasks = [
    "bace",
    "smol-property_prediction-bbbp",
    "smol-property_prediction-clintox",
    "smol-property_prediction-esol",
    "smol-property_prediction-lipo",
    "smol-property_prediction-hiv",
    "smol-property_prediction-sider",
    "qm9_homo",
    "qm9_lumo",
    "qm9_homo_lumo_gap",
    "chebi-20-text2mol",
    "chebi-20-mol2text",
    "smol-molecule_generation",
    "smol-molecule_captioning",
    "reagent_prediction",
    "forward_reaction_prediction",
    "smol-forward_synthesis",
    "retrosynthesis",
    "smol-retrosynthesis",
]

dump_dir = '/text-mol/Mol-LLM/prediction_dump'

string_only_paths = {}
# classification
string_only_paths['bace'] = '/data/all_checkpoints/bace_mlp_string_only_dump_0318/lightning_logs/version_0'
string_only_paths['bbbp'] = '/data/all_checkpoints/smol-property_prediction-bbbp_mlp_string_only_dump_0318/lightning_logs/version_0'
string_only_paths['clintox'] = '/data/all_checkpoints/smol-property_prediction-clintox_mlp_string_only_dump_0318/lightning_logs/version_0'
string_only_paths['hiv'] = '/data/all_checkpoints/smol-property_prediction-hiv_mlp_string_only_dump_0318/lightning_logs/version_0'
string_only_paths['sider'] = '/data/all_checkpoints/smol-property_prediction-sider_mlp_string_only_dump_0318/lightning_logs/version_0'

# regression
string_only_paths['esol'] = '/data/all_checkpoints/smol-property_prediction-esol_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['lipo'] = '/data/all_checkpoints/smol-property_prediction-lipo_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['qm9_homo'] = '/data/all_checkpoints/qm9_homo_mlp_string_only_12ep_0318/lightning_logs/version_0'

# captioning
string_only_paths['chebi-20-mol2text'] = '/data/all_checkpoints/chebi-20-mol2text_mlp_string_only_12ep_0318/lightning_logs/version_0'

# reaction
string_only_paths['forward'] = '/data/all_checkpoints/forward_reaction_prediction_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['reagent'] = '/data/all_checkpoints/reagent_prediction_mlp_string_only_12ep_0318/lightning_logs/version_0'


graph_only_paths = {}
# classification
graph_only_paths['bace'] = '/data/all_checkpoints/bace_qformer_graph_only_dump_0318/lightning_logs/version_0'
graph_only_paths['bbbp'] = '/data/all_checkpoints/smol-property_prediction-bbbp_qformer_graph_only_dump_0318/lightning_logs/version_0'
graph_only_paths['clintox'] = '/data/all_checkpoints/smol-property_prediction-clintox_qformer_graph_only_dump_0318/lightning_logs/version_0'
graph_only_paths['hiv'] = '/data/all_checkpoints/smol-property_prediction-hiv_qformer_graph_only_dump_0318/lightning_logs/version_0'
graph_only_paths['sider'] = '/data/all_checkpoints/smol-property_prediction-sider_qformer_graph_only_dump_0318/lightning_logs/version_0'

# regression
graph_only_paths['esol'] = '/data/all_checkpoints/smol-property_prediction-esol_qformer_graph_only_12ep_0318/lightning_logs/version_0'
graph_only_paths['lipo'] = '/data/all_checkpoints/smol-property_prediction-lipo_qformer_graph_only_12ep_0318/lightning_logs/version_0'
graph_only_paths['qm9_homo'] = '/data/all_checkpoints/qm9_homo_qformer_graph_only_12ep_0318/lightning_logs/version_0'


In [105]:
import os
import json
import re
import selfies as sf
from rdkit import Chem
from data_utils import CLASSIFICATION_BENCHMARKS, REGRESSION_BENCHMARKS, REACTION_BENCHMARKS, MOL2TEXT_BENCHMARKS

def save_scored_dump(path, tag, data_dir='/text-mol/Mol-LLM/prediction_dump'):
    dump_data = get_prediction_dump(path)
    task = dump_data[0]['task']
    if task in CLASSIFICATION_BENCHMARKS:
        error_scored_data = rank_error_classification(dump_data)
    elif task in REGRESSION_BENCHMARKS:
        error_scored_data = rank_error_regression(dump_data)
    elif task in REACTION_BENCHMARKS:
        pass
    elif task in MOL2TEXT_BENCHMARKS:
        pass
    else:
        raise ValueError
    
    with open(os.path.join(dump_dir, f'dump_{tag}_{task}.json'), 'w') as f:
        json.dump(error_scored_data, f, indent=4)

def get_prediction_dump(path):
    prediction_files = [f for f in os.listdir(path) if (f.startswith('ft-') or f.startswith('test')) and f.endswith('.json')]
    output_files = [f for f in prediction_files if 'output' in f]

    # read the output files
    output_data = []
    for p in output_files:
        with open(os.path.join(path, p), 'r') as f:
            data = json.load(f)
            output_data.extend(data)

    processed_data = []
    truncated_idx = []

    for i in range(len(output_data)):
        instance = output_data[i]
        prediction = instance['prediction']
        target = instance['target']
        task = instance['task']
        prompt = instance['prompt']
        selfies_pattern = r"(?<=<SELFIES>).*(?=</SELFIES>)"
        try:
            if "input_mol_strings" in instance.keys():
                selfies_string = re.search(selfies_pattern, instance["input_mol_strings"]).group().replace(" ", "")
            else:
                selfies_string = re.search(selfies_pattern, prompt).group().replace(" ", "")
            out = {
                'selfies' : selfies_string,
                'prediction': prediction,
                'target': target,
                'task': task
            }
            if 'prob' in instance.keys():
                out['prob'] = instance['prob']
            processed_data.append(out)
        except:
            truncated_idx.append(i)
    # print the truncated ratio
    print(f"Truncated ratio: {len(truncated_idx) / len(output_data)}")
    return processed_data

def convert_string2number(text):
    text = text.replace("<FLOAT>", "").replace("</FLOAT>", "")
    text = text.replace("<", "").replace(">", "").replace("|", "").replace(" ", "").replace("/s", "")
    return float(text)

def rank_error_regression(data):
    for i in range(len(data)):
        prediciton = convert_string2number(data[i]['prediction'])
        target = convert_string2number(data[i]['target'])
        data[i]['prediction'] = prediciton
        data[i]['target'] = target
        data[i]['error_score'] = abs(prediciton - target)
    #  sort the data by mae, and add mae rank to the data
    sorted_data = sorted(data, key=lambda x: x['error_score'])
    for i in range(len(sorted_data)):
        sorted_data[i]['error_rank'] = i + 1
    return sorted_data

def rank_error_classification(data):
    for i in range(len(data)):
        prob_posive = data[i]['prob'][1]
        if "true" in data[i]['target'].lower():
            target = 1
        elif "false" in data[i]['target'].lower():
            target = 0
        else:
            raise ValueError("Target is not true or false")
        data[i]['error_score'] = abs(target - prob_posive)
    #  sort the data by mae, and add mae rank to the data
    sorted_data = sorted(data, key=lambda x: x['error_score'])
    for i in range(len(sorted_data)):
        sorted_data[i]['error_rank'] = i + 1
    return sorted_data

In [109]:
list_path = graph_only_paths
for i in [
    'bbbp',
    'clintox',
    'hiv',
    'sider',
    'esol',
    'lipo',
    'qm9_homo'
]:
    save_scored_dump(list_path[i], tag='graph_only')

Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0


In [110]:
list_path = string_only_paths
for i in [
    'bace',
    'bbbp',
    'clintox',
    'hiv',
    'sider',
    'esol',
    'lipo',
    'qm9_homo'
]:
    save_scored_dump(list_path[i], tag='string_only')

Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0


In [107]:
string_graph_paths = {}
# classification
string_graph_paths['bace'] = '/data/all_checkpoints/MT_bace_string+graph_dump_0318/lightning_logs/version_0'
string_graph_paths['bbbp'] = '/data/all_checkpoints/MT_smol-property_prediction-bbbp_string+graph_dump_0318/lightning_logs/version_0'
string_graph_paths['clintox'] = '/data/all_checkpoints/MT_smol-property_prediction-clintox_string+graph_dump_0318/lightning_logs/version_0'
string_graph_paths['hiv'] = '/data/all_checkpoints/MT_smol-property_prediction-hiv_string+graph_dump_0318/lightning_logs/version_0'
string_graph_paths['sider'] = '/data/all_checkpoints/MT_smol-property_prediction-sider_string+graph_dump_0318/lightning_logs/version_0'

# regression
string_graph_paths['esol'] = '/data/all_checkpoints/MT_smol-property_prediction-esol_string+graph_dump_0318/lightning_logs/version_0'
string_graph_paths['lipo'] = '/data/all_checkpoints/MT_smol-property_prediction-lipo_string+graph_dump_0318/lightning_logs/version_0'
string_graph_paths['qm9_homo'] = '/data/all_checkpoints/MT_qm9_homo_string+graph_dump_0318/lightning_logs/version_0'

list_path = string_graph_paths
for i in [
    'bace',
    'bbbp',
    'clintox',
    'hiv',
    'sider',
    'esol',
    'lipo',
    'qm9_homo'
]:
    save_scored_dump(list_path[i], tag='string+graph')

Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
